# S10 화자 필드와 값 사전 확인

목적: S10이 코퍼스 문서에 붙인 화자 필드(`questioners`·`answerers`)와 값 사전(`meta_dict.json`)의 품질을 보고, 이름 정규화를 할지 정하는 근거로 쓴다(`docs/slices/10-speaker-fields.md`). 화자 이름·직위는 공개 회의록의 발언자 정보다. 질의 원문은 읽지 않는다.

이 노트북은 이름 정규화(D-20)를 넣기 전 코퍼스로 실행했다. 정규화 규칙을 정한 근거 자료다.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

In [2]:
ROOT = Path.cwd().parent
CORPUS = ROOT / "data/processed/corpus_context.jsonl"
META_DICT = ROOT / "data/processed/meta_dict.json"
ROLES = ["questioners", "answerers"]
KEYS = ["name", "position", "affiliation"]

## 데이터 로딩

In [3]:
with CORPUS.open(encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]
people = pd.DataFrame([
    {"doc_id": d["doc_id"], "role": role, **p} for d in docs for role in ROLES for p in d[role]
])
print(len(docs), people.shape)
people.groupby("role").size()

38516 (77483, 5)


role
answerers      38785
questioners    38698
dtype: int64

## 빈 값과 문서당 화자 수

항목(문서 안에서 중복을 뺀 화자) 기준 빈 값 비율이다.

In [4]:
people.assign(**{k: people[k].eq("") for k in KEYS}).groupby("role")[KEYS].mean().round(3)

,name,position,affiliation
role,,,
answerers,0.0,0.003,0.181
questioners,0.0,0.002,0.991


In [5]:
per_doc = pd.DataFrame({role: [len(d[role]) for d in docs] for role in ROLES})
per_doc.apply(lambda s: s.value_counts().sort_index().head(8))

,questioners,answerers
0,2.0,2.0
1,38350.0,38268.0
2,149.0,226.0
3,13.0,16.0
4,1.0,3.0
5,NaN,1.0
7,1.0,NaN


In [6]:
print("두 필드가 모두 빈 문서:", int((per_doc.sum(axis=1) == 0).sum()))
n_qa = pd.Series([d["n_qa"] for d in docs])
print("n_qa 대비 질의자 수 평균:", round((per_doc["questioners"] / n_qa).mean(), 3))

두 필드가 모두 빈 문서: 2
n_qa 대비 질의자 수 평균: 0.991


## 이름 형식

이름이 아닌 것이 섞였는지 본다. 길이, 한글 밖 문자, 공백, 괄호, 직함으로 끝나는 값.

In [7]:
names = people.loc[people["name"] != "", ["role", "name"]].drop_duplicates()
print(len(names), names["name"].nunique())
names["name"].str.len().value_counts().sort_index()

7260 6585


name
2      142
3     6891
4       17
5       10
6       95
7       10
8       12
9       13
10       5
11      13
12       4
13      11
14       7
15      13
16       4
17      10
18       1
19       1
29       1
Name: count, dtype: int64

In [8]:
TITLES = ("위원", "의원", "장관", "위원장", "의장", "차관", "청장", "총리", "대표", "간사")
patterns = {
    "한글 밖 문자": names["name"].str.contains(r"[^가-힣]"),
    "공백": names["name"].str.contains(" "),
    "괄호": names["name"].str.contains(r"[()（）]"),
    "직함으로 끝남": names["name"].str.endswith(TITLES) & names["name"].str.len().ge(4),
    "한 글자": names["name"].str.len().eq(1),
}
pd.DataFrame({k: [int(v.sum())] for k, v in patterns.items()})

,한글 밖 문자,공백,괄호,직함으로 끝남,한 글자
0,205,197,5,89,0


In [9]:
for k, mask in patterns.items():
    print(k, names.loc[mask, "name"].value_counts().head(8).to_dict())

한글 밖 문자 {'백혜련 위원': 2, '이수진(비)': 2, '김근태 위원': 1, '조달청장 김성호': 1, '김영진 위원': 1, '한국컨테이너부두공단이사장 조남일': 1, '김학송 위원': 1, '중소기업유통센터대표이사 이승웅': 1}
공백 {'백혜련 위원': 2, '김근태 위원': 1, '조달청장 김성호': 1, '김영진 위원': 1, '한국컨테이너부두공단이사장 조남일': 1, '김학송 위원': 1, '중소기업유통센터대표이사 이승웅': 1, '김기춘 위원': 1}
괄호 {'이수진(비)': 2, '최경환(국)': 1, '최경환(한)': 1, '최경환(평)': 1}
직함으로 끝남 {'백혜련 위원': 2, '김근태 위원': 1, '김영진 위원': 1, '김학송 위원': 1, '김기춘 위원': 1, '이인기 위원': 1, '이주영 위원': 1, '이재오 위원': 1}
한 글자 {}


## 같은 이름, 여러 직위·소속

동명이인이거나 직위가 바뀐 경우다. 이름만으로 조건을 대조하면 둘을 구분하지 못한다.

In [10]:
variants = people[people["name"] != ""].groupby("name")[["position", "affiliation"]].nunique()
print("이름 수:", len(variants), "직위 2개 이상:", int((variants["position"] > 1).sum()),
      "소속 2개 이상:", int((variants["affiliation"] > 1).sum()))
variants.sort_values("position", ascending=False).head(5)

이름 수: 6585 직위 2개 이상: 2380 소속 2개 이상: 1840


,position,affiliation
name,,
추경호,13,8
김성호,12,9
이상훈,12,10
김재원,11,5
이종구,10,7


In [11]:
print("직위 값 상위:", people["position"].value_counts().head(15).to_dict())

직위 값 상위: {'위원': 34221, '차관': 4272, '장관': 4205, '소위원장': 2970, '청장': 1997, '제1차관': 1484, '제2차관': 1196, '의원': 1166, '사장': 1104, '위원장': 1083, '차장': 954, '국무총리': 871, '이사장': 867, '진술인': 850, '원장': 840}


## 사전 크기

In [12]:
meta_dict = json.loads(META_DICT.read_text(encoding="utf-8"))
pd.Series({k: len(v) for k, v in meta_dict.items()})

affiliation       1551
committee_name     497
meeting_name         5
name              6585
position          2419
dtype: int64

In [13]:
for k in ["committee_name", "meeting_name"]:
    print(k, Counter(meta_dict[k]).most_common(8))
print("이름 문서 수 분포:", pd.Series(meta_dict["name"]).describe().round(1).to_dict())

committee_name [('예산결산특별위원회', 2053), ('법제사법위원회', 1370), ('국회본회의', 1159), ('환경노동위원회', 986), ('농림축산식품해양수산위원회', 954), ('정무위원회', 926), ('국방위원회', 896), ('보건복지위원회', 856)]
meeting_name [('국정감사', 15884), ('소위원회', 15553), ('특별위원회', 3867), ('예산결산특별위원회', 2053), ('본회의', 1159)]
이름 문서 수 분포: {'count': 6585.0, 'mean': 11.8, 'std': 28.2, 'min': 1.0, '25%': 1.0, '50%': 2.0, '75%': 7.0, 'max': 442.0}


## 정규화 후보 규칙의 효과

후보: (1) 끝의 괄호 표기(`(비)` 등)를 지운다. (2) 공백이 있으면 마지막 토막이 직함이면 첫 토막을, 아니면 마지막 토막을 이름으로 둔다(`○○○ 위원` → `○○○`, `조달청장 ○○○` → `○○○`). 규칙을 적용했을 때 바뀌는 값과 결과가 2\~4자 한글인지 본다.

In [14]:
def normalize(name):
    name = re.sub(r"\s*[(（][^)）]*[)）]$", "", name).strip()
    if " " in name:
        parts = name.split()
        name = parts[0] if parts[-1].endswith(TITLES) else parts[-1]
    return name


changed = names.assign(new=names["name"].map(normalize)).query("name != new")
print("바뀌는 값:", len(changed), "/", len(names))
print("결과가 2~4자 한글이 아닌 것:", int((~changed["new"].str.fullmatch(r"[가-힣]{2,4}")).sum()))
changed.sample(min(10, len(changed)), random_state=0)[["name", "new"]]

바뀌는 값: 202 / 7260
결과가 2~4자 한글이 아닌 것: 2


,name,new
1296,김동욱 위원,김동욱
62586,김웅 위원,김웅
22632,박명재 위원,박명재
16994,경기도지사 남경필,남경필
63500,권영세 위원,권영세
67135,서범수 위원,서범수
567,중소기업유통센터대표이사 이승웅,이승웅
52496,김현아 위원,김현아
920,이재오 위원,이재오
56893,병무청장 기찬수,기찬수


In [15]:
rest = names.loc[~names["name"].map(normalize).str.fullmatch(r"[가-힣]{2,4}"), "name"]
print("규칙 뒤에도 2~4자 한글이 아닌 이름:", len(rest))
rest.value_counts().head(10).to_dict()

규칙 뒤에도 2~4자 한글이 아닌 이름: 13


{'존필메리디스': 1,
 '사미르에이투바이엡': 1,
 '\\김황식': 1,
 '데이비드 스트라우브': 1,
 '류지영위원': 1,
 '요하네스타머': 1,
 '대법원장후보자김명수': 1,
 '최연혜 위': 1,
 '000': 1,
 '통일부장관': 1}

## 관찰

- 화자 항목은 질의자 38,698개, 응답자 38,785개다. 문서 대부분(질의자 38,350, 응답자 38,268)은 화자가 한 명씩이다. 두 필드가 모두 빈 문서는 2개이고, 원천 라벨에 질의자 ID만 있다
- 이름 빈 값은 0이다. 소속은 질의자 99.1%, 응답자 18.1%가 비어 있어 질의자 소속은 조건으로 쓸 수 없다
- 이름 값 7,260개(역할별 중복 제거) 중 202개(2.8%)는 이름이 아닌 것이 붙어 있다. `○○○ 위원`처럼 직함이 뒤에 붙거나, `조달청장 ○○○`처럼 기관·직위가 앞에 붙거나, `○○○(비)`처럼 괄호 표기가 붙은 경우다. 이대로 두면 질의의 이름과 대조할 때 빠진다
- 정규화 후보 규칙(괄호 표기 삭제, 공백이 있으면 직함 앞 토막이나 마지막 토막)을 적용하면 202개가 바뀌고, 그중 2개를 뺀 결과가 2\~4자 한글 이름이 된다. 규칙 뒤에도 남는 13개는 외국인 이름, 붙여 쓴 직함(`○○○위원`), `000` 같은 값이다
- 이름 6,585개 중 2,380개가 직위 2개 이상으로 나온다(장관→총리처럼 직위가 바뀐 경우와 동명이인). 이름만으로 대조하면 동명이인을 구분하지 못한다. 사전에는 이름별 문서 수만 둔다
- 사전 크기: 위원회 497, 회의 종류 5, 이름 6,585, 직위 2,419, 소속 1,551
